## Preliminaries


In [1]:
from os.path import abspath
from subprocess import run

import meeko
import molscrub
import prody
import pandas as pd
from rdkit import Chem

from src.utils.prep import ligand_prep


In [2]:
CWD = abspath(".")


In [3]:
nat_lig = pd.read_csv(
        "../data/antibiotics_pubchem_query.csv", index_col=0
    )
mir_lig = pd.read_csv("../data/mirror_ligand.csv", index_col=0)


In [4]:
nat_lig.head()


,amoxicillin,piperacillin,ticarcillin,cefepime,cefazolin,ceftolozane,ceftriaxone,ceftazidime,aztreonam,imipenem,...,linezolid,amikacin,tobramycin,streptomycin,kanamycin,trimethoprim,sulfamethoxazole,ciprofloxacin,levofloxacin,Moxifloxacin
cid,33613,43672,36921,5479537,33255,53234134,5479530,5481173,5742832,104838,...,441401,37768,36294,19649,6032,5578,5329,2764,149096,152946
smiles,CC1([C@@H](N2[C@H](S1)[C@@H](C2=O)NC(=O)[C@@H]...,CCN1CCN(C(=O)C1=O)C(=O)N[C@H](C2=CC=CC=C2)C(=O...,CC1([C@@H](N2[C@H](S1)[C@@H](C2=O)NC(=O)[C@@H]...,C[N+]1(CCCC1)CC2=C(N3[C@@H]([C@@H](C3=O)NC(=O)...,CC1=NN=C(S1)SCC2=C(N3[C@@H]([C@@H](C3=O)NC(=O)...,CC(C)(C(=O)O)O/N=C(/C1=NSC(=N1)N)\C(=O)N[C@H]2...,CN1C(=NC(=O)C(=O)N1)SCC2=C(N3[C@@H]([C@@H](C3=...,CC(C)(C(=O)O)O/N=C(/C1=CSC(=N1)N)\C(=O)N[C@H]2...,C[C@H]1[C@@H](C(=O)N1S(=O)(=O)O)NC(=O)/C(=N\OC...,C[C@H]([C@@H]1[C@H]2CC(=C(N2C1=O)C(=O)O)SCCN=CN)O,...,CC(=O)NC[C@H]1CN(C(=O)O1)C2=CC(=C(C=C2)N3CCOCC3)F,C1[C@@H]([C@H]([C@@H]([C@H]([C@@H]1NC(=O)[C@H]...,C1[C@@H]([C@H]([C@@H]([C@H]([C@@H]1N)O[C@@H]2[...,C[C@H]1[C@@]([C@H]([C@@H](O1)O[C@@H]2[C@H]([C@...,C1[C@H]([C@@H]([C@H]([C@@H]([C@H]1N)O[C@@H]2[C...,COC1=CC(=CC(=C1OC)OC)CC2=CN=C(N=C2N)N,CC1=CC(=NO1)NS(=O)(=O)C2=CC=C(C=C2)N,C1CC1N2C=C(C(=O)C3=CC(=C(C=C32)N4CCNCC4)F)C(=O)O,C[C@H]1COC2=C3N1C=C(C(=O)C3=CC(=C2N4CCN(CC4)C)...,COC1=C2C(=CC(=C1N3C[C@@H]4CCCN[C@@H]4C3)F)C(=O...
inchi,InChI=1S/C16H19N3O5S/c1-16(2)11(15(23)24)19-13...,InChI=1S/C23H27N5O7S/c1-4-26-10-11-27(19(32)18...,InChI=1S/C15H16N2O6S2/c1-15(2)9(14(22)23)17-11...,InChI=1S/C19H24N6O5S2/c1-25(5-3-4-6-25)7-10-8-...,InChI=1S/C14H14N8O4S3/c1-6-17-18-14(29-6)28-4-...,"InChI=1S/C23H30N12O8S2/c1-23(2,20(40)41)43-31-...",InChI=1S/C18H18N8O7S3/c1-25-18(22-12(28)13(29)...,"InChI=1S/C22H22N6O7S2/c1-22(2,20(33)34)35-26-1...","InChI=1S/C13H17N5O8S2/c1-5-7(10(20)18(5)28(23,...",InChI=1S/C12H17N3O4S/c1-6(16)9-7-4-8(20-3-2-14...,...,InChI=1S/C16H20FN3O4/c1-11(21)18-9-13-10-20(16...,InChI=1S/C22H43N5O13/c23-2-1-8(29)20(36)27-7-3...,InChI=1S/C18H37N5O9/c19-3-9-8(25)2-7(22)17(29-...,"InChI=1S/C21H39N7O12/c1-5-21(36,4-30)16(40-17-...",InChI=1S/C18H36N4O11/c19-2-6-10(25)12(27)13(28...,InChI=1S/C14H18N4O3/c1-19-10-5-8(6-11(20-2)12(...,InChI=1S/C10H11N3O3S/c1-7-6-10(12-16-7)13-17(1...,InChI=1S/C17H18FN3O3/c18-13-7-11-14(8-15(13)20...,InChI=1S/C18H20FN3O4/c1-10-9-26-17-14-11(16(23...,InChI=1S/C21H24FN3O4/c1-29-20-17-13(19(26)14(2...
coordinate_type,2d,2d,2d,2d,2d,2d,2d,2d,2d,2d,...,2d,2d,2d,2d,2d,2d,2d,2d,2d,2d
xlogp,-2,0.5,0.8,-0.1,-0.4,-3.2,-1.3,0.4,0.3,-0.7,...,0.7,-7.9,-6.2,-8,-6.9,0.9,0.9,-1.1,-0.4,0.6


In [5]:
mir_lig.head()


,amoxicillin,piperacillin,ticarcillin,cefepime,cefazolin,ceftolozane,ceftriaxone,ceftazidime,aztreonam,imipenem,...,linezolid,amikacin,tobramycin,streptomycin,kanamycin,trimethoprim,sulfamethoxazole,ciprofloxacin,levofloxacin,Moxifloxacin
cid,33613,43672,36921,5479537,33255,53234134,5479530,5481173,5742832,104838,...,441401,37768,36294,19649,6032,5578,5329,2764,149096,152946
smiles,CC1([C@@H](N2[C@H](S1)[C@@H](C2=O)NC(=O)[C@@H]...,CCN1CCN(C(=O)C1=O)C(=O)N[C@H](C2=CC=CC=C2)C(=O...,CC1([C@@H](N2[C@H](S1)[C@@H](C2=O)NC(=O)[C@@H]...,C[N+]1(CCCC1)CC2=C(N3[C@@H]([C@@H](C3=O)NC(=O)...,CC1=NN=C(S1)SCC2=C(N3[C@@H]([C@@H](C3=O)NC(=O)...,CC(C)(C(=O)O)O/N=C(/C1=NSC(=N1)N)\C(=O)N[C@H]2...,CN1C(=NC(=O)C(=O)N1)SCC2=C(N3[C@@H]([C@@H](C3=...,CC(C)(C(=O)O)O/N=C(/C1=CSC(=N1)N)\C(=O)N[C@H]2...,C[C@H]1[C@@H](C(=O)N1S(=O)(=O)O)NC(=O)/C(=N\OC...,C[C@H]([C@@H]1[C@H]2CC(=C(N2C1=O)C(=O)O)SCCN=CN)O,...,CC(=O)NC[C@H]1CN(C(=O)O1)C2=CC(=C(C=C2)N3CCOCC3)F,C1[C@@H]([C@H]([C@@H]([C@H]([C@@H]1NC(=O)[C@H]...,C1[C@@H]([C@H]([C@@H]([C@H]([C@@H]1N)O[C@@H]2[...,C[C@H]1[C@@]([C@H]([C@@H](O1)O[C@@H]2[C@H]([C@...,C1[C@H]([C@@H]([C@H]([C@@H]([C@H]1N)O[C@@H]2[C...,COC1=CC(=CC(=C1OC)OC)CC2=CN=C(N=C2N)N,CC1=CC(=NO1)NS(=O)(=O)C2=CC=C(C=C2)N,C1CC1N2C=C(C(=O)C3=CC(=C(C=C32)N4CCNCC4)F)C(=O)O,C[C@H]1COC2=C3N1C=C(C(=O)C3=CC(=C2N4CCN(CC4)C)...,COC1=C2C(=CC(=C1N3C[C@@H]4CCCN[C@@H]4C3)F)C(=O...
inchi,InChI=1S/C16H19N3O5S/c1-16(2)11(15(23)24)19-13...,InChI=1S/C23H27N5O7S/c1-4-26-10-11-27(19(32)18...,InChI=1S/C15H16N2O6S2/c1-15(2)9(14(22)23)17-11...,InChI=1S/C19H24N6O5S2/c1-25(5-3-4-6-25)7-10-8-...,InChI=1S/C14H14N8O4S3/c1-6-17-18-14(29-6)28-4-...,"InChI=1S/C23H30N12O8S2/c1-23(2,20(40)41)43-31-...",InChI=1S/C18H18N8O7S3/c1-25-18(22-12(28)13(29)...,"InChI=1S/C22H22N6O7S2/c1-22(2,20(33)34)35-26-1...","InChI=1S/C13H17N5O8S2/c1-5-7(10(20)18(5)28(23,...",InChI=1S/C12H17N3O4S/c1-6(16)9-7-4-8(20-3-2-14...,...,InChI=1S/C16H20FN3O4/c1-11(21)18-9-13-10-20(16...,InChI=1S/C22H43N5O13/c23-2-1-8(29)20(36)27-7-3...,InChI=1S/C18H37N5O9/c19-3-9-8(25)2-7(22)17(29-...,"InChI=1S/C21H39N7O12/c1-5-21(36,4-30)16(40-17-...",InChI=1S/C18H36N4O11/c19-2-6-10(25)12(27)13(28...,InChI=1S/C14H18N4O3/c1-19-10-5-8(6-11(20-2)12(...,InChI=1S/C10H11N3O3S/c1-7-6-10(12-16-7)13-17(1...,InChI=1S/C17H18FN3O3/c18-13-7-11-14(8-15(13)20...,InChI=1S/C18H20FN3O4/c1-10-9-26-17-14-11(16(23...,InChI=1S/C21H24FN3O4/c1-29-20-17-13(19(26)14(2...
coordinate_type,2d,2d,2d,2d,2d,2d,2d,2d,2d,2d,...,2d,2d,2d,2d,2d,2d,2d,2d,2d,2d
mirror_smi,CC1(C)S[C@H]2[C@@H](NC(=O)[C@@H](N)c3ccc(O)cc3...,CCN1CCN(C(=O)N[C@H](C(=O)N[C@H]2C(=O)N3[C@H]2S...,CC1(C)S[C@H]2[C@@H](NC(=O)[C@@H](C(=O)O)c3ccsc...,CO/N=C(\C(=O)N[C@H]1C(=O)N2C(C(=O)[O-])=C(C[N+...,Cc1nnc(SCC2=C(C(=O)O)N3C(=O)[C@H](NC(=O)Cn4cnn...,Cn1c(N)c(NC(=O)NCCN)c[n+]1CC1=C(C(=O)[O-])N2C(...,CO/N=C(\C(=O)N[C@H]1C(=O)N2C(C(=O)O)=C(CSc3nc(...,CC(C)(O/N=C(\C(=O)N[C@H]1C(=O)N2C(C(=O)[O-])=C...,C[C@@H]1[C@@H](NC(=O)/C(=N\OC(C)(C)C(=O)O)c2cs...,C[C@H](O)[C@@H]1C(=O)N2C(C(=O)O)=C(SCCN=CN)C[C...,...,CC(=O)NC[C@@H]1CN(c2ccc(N3CCOCC3)c(F)c2)C(=O)O1,NCC[C@@H](O)C(=O)N[C@H]1C[C@@H](N)[C@H](O[C@@H...,NC[C@@H]1O[C@@H](O[C@@H]2[C@@H](O)[C@H](O[C@@H...,CN[C@H]1[C@@H](O[C@@H]2[C@@H](O[C@@H]3[C@@H](O...,NC[C@@H]1O[C@@H](O[C@@H]2[C@@H](O)[C@H](O[C@@H...,COc1cc(Cc2cnc(N)nc2N)cc(OC)c1OC,Cc1cc(NS(=O)(=O)c2ccc(N)cc2)no1,O=C(O)c1cn(C2CC2)c2cc(N3CCNCC3)c(F)cc2c1=O,C[C@@H]1COc2c(N3CCN(C)CC3)c(F)cc3c(=O)c(C(=O)O...,COc1c(N2C[C@H]3CCCN[C@H]3C2)c(F)cc2c(=O)c(C(=O...


## Ligand Preparation

In [ ]:
#* Performed successfully 4:52:24 AM for 8m 0.9s
# for lg, ml in zip(nat_lig, mir_lig):
#     lg_name: str = f"Native {lg.title()}"
#     ml_name: str = f"Mirror {ml.title()}"
#
#     lg_smi: str = nat_lig[lg]["smiles"]
#     ml_smi: str = mir_lig[ml]["mirror_smi"]
#     ligand_prep(
#         lg_smi, lg_name, f"{CWD}/../data/ligand_prep"
#     )
#     ligand_prep(
#         ml_smi, ml_name, f"{CWD}/../data/ligand_prep"
#     )


Scrubing Native Amoxicillin with scrub.py CC1([C@@H](N2[C@H](S1)[C@@H](C2=O)NC(=O)[C@@H](C3=CC=C(C=C3)O)N)C(=O)O)C -o /var/home/aeersriv/Documents/antibiotics_mirror_life/comp_code-antibiotics_mirror_life/notebooks/../data/ligand_prep/Native Amoxicillin.sdf --ph 7.0 --skip_tautomer --skip_acidbase
Scrub completed.
Summary of what happened:
Input molecules supplied: 1
mols processed: 1, skipped by rdkit: 0, failed: 0
nr isomers (tautomers and acid/base conjugates): 1 (avg. 1.000 per mol)
nr conformers:  1 (avg. 1.000 per isomer, 1.000 per mol)
Preparing Native Amoxicillin with mk_prepare_ligand.py -i /var/home/aeersriv/Documents/antibiotics_mirror_life/comp_code-antibiotics_mirror_life/notebooks/../data/ligand_prep/Native Amoxicillin.sdf -o /var/home/aeersriv/Documents/antibiotics_mirror_life/comp_code-antibiotics_mirror_life/notebooks/../data/ligand_prep/Native Amoxicillin.pdbqt
Input molecules processed: 1, skipped: 0
PDBQT files written: 1
PDBQT files not written due to error: 0
Inpu

## Target Preparation

In [6]:
target_info: dict[str, str] = {
        "6I1H": "MER",
        "5M18": "MUR",
        "8CF1": "TAC",
        "8CGD": "AM2",
        "3FL9": "TOP",
        "1TX2": "680",
        "4DUH": "RLI",
        "5IQB": "GNP"
    }


In [7]:
receptor_info: dict = {}


In [11]:
def receptor_atoms(pdb_token: str, path: str, outfile: str):
    atoms_from_pdb = prody.parsePDB(f"{path}/{pdb_token}.pdb")
    receptor_selection = "chain A and not water and not hetero"
    receptor_atoms = atoms_from_pdb.select(receptor_selection)
    prody_receptorPDB = f"{outfile}/{pdb_token}_receptor_atoms.pdb"
    prody.writePDB(prody_receptorPDB, receptor_atoms)
    return prody_receptorPDB


def prep_receptor(pdb_token: str, path: str, ligand: str, outfile: str, rc_hash: dict, mirror: bool):
    atoms_from_pdb = prody.parsePDB(f"{path}/{pdb_token}.pdb")
    ligand_selection = f"chain A and resname {ligand}"
    ligand_atoms = atoms_from_pdb.select(ligand_selection)
    center_x, center_y, center_z = prody.calcCenter(ligand_atoms)
    center = [center_x, center_y, center_z]

    if mirror:
        prody_ligandPDB = f"{outfile}/L-{ligand}.pdb"
    else:
        prody_ligandPDB = f"{outfile}/{ligand}.pdb"

    prody.writePDB(prody_ligandPDB, ligand_atoms)


    size_x = 20.0
    size_y = 20.0
    size_z = 20.0
    size = [size_x, size_y, size_z]

    rc_hash[pdb_token] = {
            "size": size,
            "center": center
        }

    file_name = f"{outfile}/{pdb_token}_prep"
    prody_receptorPDB = receptor_atoms(pdb_token, path, outfile)
    mkpreprecep: list = [
            "mk_prepare_receptor.py",
            "-i", f"{prody_receptorPDB}",
            "-o", f"{file_name}",
            "-p", "--write_pdb", f"{file_name}.pdb",
            "-v", "--box_center",
            str(center_x), str(center_y), str(center_z),
            "--box_size", str(size_x), str(size_y), str(size_z),
            "--allow_bad_res",
            "--default_altloc"
        ]
    print(" ".join(mkpreprecep))
    run(mkpreprecep)


In [ ]:
#! this is a test code
# prep_receptor(
#     "6I1H".lower(),
#     f"../data/targets",
#     "MER",
#     f"{CWD}/../data/receptor_prep"
# )


@> 5241 atoms and 1 coordinate set(s) were parsed in 0.03s.
@> 5241 atoms and 1 coordinate set(s) were parsed in 0.04s.


mk_prepare_receptor.py -i /var/home/aeersriv/Documents/antibiotics_mirror_life/comp_code-antibiotics_mirror_life/notebooks/../data/receptor_prep/6i1h_receptor_atoms.pdb -o /var/home/aeersriv/Documents/antibiotics_mirror_life/comp_code-antibiotics_mirror_life/notebooks/../data/receptor_prep/6i1h_prep -p --write_pdb /var/home/aeersriv/Documents/antibiotics_mirror_life/comp_code-antibiotics_mirror_life/notebooks/../data/receptor_prep/6i1h_prep.pdb -v --box_center -5.978153846153846 -52.50296153846154 38.259846153846155 --box_size 20.0 20.0 20.0


@> 2519 atoms and 1 coordinate set(s) were parsed in 0.01s.



Files written:
    /var/home/aeersriv/Documents/antibiotics_mirror_life/comp_code-antibiotics_mirror_life/notebooks/../data/receptor_prep/6i1h_prep.pdb <-- processed receptor PDB
  /var/home/aeersriv/Documents/antibiotics_mirror_life/comp_code-antibiotics_mirror_life/notebooks/../data/receptor_prep/6i1h_prep.pdbqt <-- static (i.e., rigid) receptor input file
/var/home/aeersriv/Documents/antibiotics_mirror_life/comp_code-antibiotics_mirror_life/notebooks/../data/receptor_prep/6i1h_prep.box.txt <-- Vina-style box dimension file
/var/home/aeersriv/Documents/antibiotics_mirror_life/comp_code-antibiotics_mirror_life/notebooks/../data/receptor_prep/6i1h_prep.box.pdb <-- PDB file to visualize the grid box


In [ ]:
#! this is a test code
# prep_receptor(
#     f"L-{'6I1H'.lower()}",
#     f"../data/mirror_targets",
#     "MER",
#     f"{CWD}/../data/receptor_prep"
# )


@> 5241 atoms and 1 coordinate set(s) were parsed in 0.02s.
@> 5241 atoms and 1 coordinate set(s) were parsed in 0.02s.


mk_prepare_receptor.py -i /var/home/aeersriv/Documents/antibiotics_mirror_life/comp_code-antibiotics_mirror_life/notebooks/../data/receptor_prep/L-6i1h_receptor_atoms.pdb -o /var/home/aeersriv/Documents/antibiotics_mirror_life/comp_code-antibiotics_mirror_life/notebooks/../data/receptor_prep/L-6i1h_prep -p --write_pdb /var/home/aeersriv/Documents/antibiotics_mirror_life/comp_code-antibiotics_mirror_life/notebooks/../data/receptor_prep/L-6i1h_prep.pdb -v --box_center 5.978153846153846 52.50296153846154 -38.259846153846155 --box_size 20.0 20.0 20.0


@> 2519 atoms and 1 coordinate set(s) were parsed in 0.01s.



Files written:
    /var/home/aeersriv/Documents/antibiotics_mirror_life/comp_code-antibiotics_mirror_life/notebooks/../data/receptor_prep/L-6i1h_prep.pdb <-- processed receptor PDB
  /var/home/aeersriv/Documents/antibiotics_mirror_life/comp_code-antibiotics_mirror_life/notebooks/../data/receptor_prep/L-6i1h_prep.pdbqt <-- static (i.e., rigid) receptor input file
/var/home/aeersriv/Documents/antibiotics_mirror_life/comp_code-antibiotics_mirror_life/notebooks/../data/receptor_prep/L-6i1h_prep.box.txt <-- Vina-style box dimension file
/var/home/aeersriv/Documents/antibiotics_mirror_life/comp_code-antibiotics_mirror_life/notebooks/../data/receptor_prep/L-6i1h_prep.box.pdb <-- PDB file to visualize the grid box


In [13]:
for rc, lg in target_info.items():
    prep_receptor(
        rc.lower(),
        f"{CWD}/../data/targets",
        lg,
        f"{CWD}/../data/native_target_prep",
        receptor_info,
        False
    )
    prep_receptor(
        f"L-{rc.lower()}",
        f"{CWD}/../data/mirror_targets",
        lg,
        f"{CWD}/../data/mirror_target_prep",
        receptor_info,
        True
    )


@> 5241 atoms and 1 coordinate set(s) were parsed in 0.02s.
@> 5241 atoms and 1 coordinate set(s) were parsed in 0.02s.


mk_prepare_receptor.py -i /var/home/aeersriv/Documents/antibiotics_mirror_life/comp_code-antibiotics_mirror_life/notebooks/../data/native_target_prep/6i1h_receptor_atoms.pdb -o /var/home/aeersriv/Documents/antibiotics_mirror_life/comp_code-antibiotics_mirror_life/notebooks/../data/native_target_prep/6i1h_prep -p --write_pdb /var/home/aeersriv/Documents/antibiotics_mirror_life/comp_code-antibiotics_mirror_life/notebooks/../data/native_target_prep/6i1h_prep.pdb -v --box_center -5.978153846153846 -52.50296153846154 38.259846153846155 --box_size 20.0 20.0 20.0 --allow_bad_res --default_altloc



mk_prepare_receptor.py: error: argument --default_altloc: expected one argument
@> 5241 atoms and 1 coordinate set(s) were parsed in 0.02s.
@> 5241 atoms and 1 coordinate set(s) were parsed in 0.02s.


usage: mk_prepare_receptor.py [-h] [--read_pdb PDB_FILENAME]
                              [--read_pqr PQR_FILENAME] [-i MACROMOL_FILENAME]
                              [-o OUTPUT_BASENAME] [-p [PDBQT_FILENAME ...]]
                              [-j [JSON_FILENAME ...]]
                              [--write_pdb [PDB_FILENAME ...]]
                              [-g [GPF_FILENAME ...]]
                              [-v [VINA_BOX_FILENAME ...]]
                              [--debug_fn DEBUG_FN] [-n SET_TEMPLATE]
                              [-d DELETE_RESIDUES] [-b BLUNT_ENDS]
                              [--add_templates ADD_TEMPLATES]
                              [--cache_templates [CACHE_TEMPLATES]]
                              [--mk_config JSON_FILENAME] [-a]
                              [--default_altloc DEFAULT_ALTLOC]
                              [--wanted_altloc WANTED_ALTLOC] [-f FLEXRES]
                              [-t ROT_TERMINAL_GROUP]
                             


mk_prepare_receptor.py: error: argument --default_altloc: expected one argument
@> 10684 atoms and 1 coordinate set(s) were parsed in 0.05s.


...

usage: mk_prepare_receptor.py [-h] [--read_pdb PDB_FILENAME]
                              [--read_pqr PQR_FILENAME] [-i MACROMOL_FILENAME]
                              [-o OUTPUT_BASENAME] [-p [PDBQT_FILENAME ...]]
                              [-j [JSON_FILENAME ...]]
                              [--write_pdb [PDB_FILENAME ...]]
                              [-g [GPF_FILENAME ...]]
                              [-v [VINA_BOX_FILENAME ...]]
                              [--debug_fn DEBUG_FN] [-n SET_TEMPLATE]
                              [-d DELETE_RESIDUES] [-b BLUNT_ENDS]
                              [--add_templates ADD_TEMPLATES]
                              [--cache_templates [CACHE_TEMPLATES]]
                              [--mk_config JSON_FILENAME] [-a]
                              [--default_altloc DEFAULT_ALTLOC]
                              [--wanted_altloc WANTED_ALTLOC] [-f FLEXRES]
                              [-t ROT_TERMINAL_GROUP]
                             

@> 18957 atoms and 1 coordinate set(s) were parsed in 0.07s.


mk_prepare_receptor.py -i /var/home/aeersriv/Documents/antibiotics_mirror_life/comp_code-antibiotics_mirror_life/notebooks/../data/mirror_target_prep/L-8cf1_receptor_atoms.pdb -o /var/home/aeersriv/Documents/antibiotics_mirror_life/comp_code-antibiotics_mirror_life/notebooks/../data/mirror_target_prep/L-8cf1_prep -p --write_pdb /var/home/aeersriv/Documents/antibiotics_mirror_life/comp_code-antibiotics_mirror_life/notebooks/../data/mirror_target_prep/L-8cf1_prep.pdb -v --box_center -248.1948125 -311.26956249999995 -211.46221874999995 --box_size 20.0 20.0 20.0 --allow_bad_res --default_altloc
usage: mk_prepare_receptor.py [-h] [--read_pdb PDB_FILENAME]
                              [--read_pqr PQR_FILENAME] [-i MACROMOL_FILENAME]
                              [-o OUTPUT_BASENAME] [-p [PDBQT_FILENAME ...]]
                              [-j [JSON_FILENAME ...]]
                              [--write_pdb [PDB_FILENAME ...]]
                              [-g [GPF_FILENAME ...]]
             


mk_prepare_receptor.py: error: argument --default_altloc: expected one argument
@> 94181 atoms and 1 coordinate set(s) were parsed in 0.40s.


TypeError: atoms must be an Atomic instance

## Docking

In [62]:
ligand_targets: dict[str, list[str]] = {
        "6I1H": [
            "amoxicillin", # penicillins
            "piperacillin",
            "ticarcillin",
            "cefepime", # cephalosporins
            "cefazolin",
            "ceftolozane",
            "ceftriaxone",
            "ceftazidime",
            "aztreonam", # monobactam
            "imipenem", # carbapenems
            "meropenem",
            "doripenem"
        ],
        "5M18": [
            "amoxicillin", # penicillins
            "piperacillin",
            "ticarcillin",
            "cefepime", # cephalosporins
            "cefazolin",
            "ceftolozane",
            "ceftriaxone",
            "ceftazidime",
            "aztreonam", # monobactam
            "imipenem", # carbapenems
            "meropenem",
            "doripenem"
        ],
        "8CF1": [ # 30S subunit
            "minocycline", # tetracyclines
            "tigecycline",
            "doxycycline",
            "tetracycline",
            "Chlortetracycline",
            "amikacin", # aminoglycosides
            "tobramycin",
            "streptomycin",
            "kanamycin"
        ],
        "5IQB": [ "amikacin", # aminoglycosides
            "tobramycin",
            "streptomycin",
            "kanamycin"
        ],
        "8CGD": [ # 50S subunit
            "azithromycin", # macrolides
            "erythromycin",
            "clarithromycin",
            "chloramphenicol" # chlorampenicol
        ],
        "3FL9": [
            "trimethoprim" # trimethoprim
        ],
        "1TX2": [
            "sulfamethoxazole" # sulfamethoxazole
        ],
        "4DUH": [
            "ciprofloxacin", # quinolones
            "levofloxacin",
            "Moxifloxacin"
        ]
    }


In [63]:
receptors = ["6I1H", "8CF1", "L-6I1H", "L-8CF1"]


In [66]:
receptor_info


{'6i1h': {'size': [20.0, 20.0, 20.0],
  'center': [np.float64(-5.978153846153846),
   np.float64(-52.50296153846154),
   np.float64(38.259846153846155)]},
 'L-6i1h': {'size': [20.0, 20.0, 20.0],
  'center': [np.float64(5.978153846153846),
   np.float64(52.50296153846154),
   np.float64(-38.259846153846155)]},
 '5m18': {'size': [20.0, 20.0, 20.0],
  'center': [np.float64(19.421823529411764),
   np.float64(-18.162411764705883),
   np.float64(-52.577411764705886)]},
 'L-5m18': {'size': [20.0, 20.0, 20.0],
  'center': [np.float64(-19.421823529411764),
   np.float64(18.162411764705883),
   np.float64(52.577411764705886)]},
 '8cf1': {'size': [20.0, 20.0, 20.0],
  'center': [np.float64(248.1948125),
   np.float64(311.26956249999995),
   np.float64(211.46221874999995)]},
 'L-8cf1': {'size': [20.0, 20.0, 20.0],
  'center': [np.float64(-248.1948125),
   np.float64(-311.26956249999995),
   np.float64(-211.46221874999995)]}}

In [64]:
import vina


In [70]:
out = f"{CWD}/../data/vina"
for rc in receptors:
    exhaustiveness = 8

    rcTXT = f"{CWD}/../data/native_target_prep/{rc.lower()}_prep.box.txt"
    rcPDBQT = f"{CWD}/../data/native_target_prep/6i1h_prep.pdbqt"
    if rc.startswith("L-"):
        rcTXT = f"{CWD}/../data/mirror_target_prep/L-{rc.lower()}_prep.box.txt"
        rcPDBQT = f"{CWD}/../data/mirror_target_prep/L-6i1h_prep.pdbqt"

    for lg in ligand_targets[rc]:
        nat_ligPDBQT = f"{CWD}/../data/ligand_prep/Native_{lg.title()}.pdbqt"
        mir_ligPDBQT = f"{CWD}/../data/ligand_prep/Mirror_{lg.title()}.pdbqt"
        nat_outputPDBQT = f"{out}/{lg}-{rc}_vina.pdbqt" #@param {type:"string"}
        mir_outputPDBQT = f"{out}/Mirror_{lg}-{rc}_vina.pdbqt" #@param {type:"string"}

        v1 = vina.Vina()
        v1.set_receptor(rcPDBQT)
        v1.compute_vina_maps(
            center=receptor_info[rc.lower()]["center"],
            box_size=receptor_info[rc.lower()]["size"])
        v1.set_ligand_from_file(nat_ligPDBQT)
        v1.dock(exhaustiveness=exhaustiveness)
        v1.write_poses(nat_outputPDBQT)

        v2 = vina.Vina()
        v2.set_receptor(rcPDBQT)
        v2.compute_vina_maps(
            center=receptor_info[f"L-{rc.lower()}"]["center"],
            box_size=receptor_info[f"L-{rc.lower()}"]["size"]
        )
        v2.set_ligand_from_file(mir_ligPDBQT)
        v2.dock(exhaustiveness=exhaustiveness)
        v2.write_poses(mir_outputPDBQT)


Computing Vina grid ... done.
Performing docking (random seed: -1881765891) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1       -7.889          0          0
   2       -7.888      3.418      4.722
   3       -7.723      3.271      8.704
   4       -7.707      1.797      2.334
   5       -7.305      5.381      6.826
   6       -7.297      3.607      5.382
   7       -7.257      2.766      4.563
   8       -7.219      5.306      10.89
   9       -7.189       5.56       10.4
  10       -6.943      3.815      5.781
  11       -6.789      5.065      10.73
  12       -6.614      5.168      11.28
  13       -6.558      6.973      9.672
  14       -6.556       3.29      4.971
  15       -6.522      5.169      9.628
  16       -6.447      4.521      9.5

Performing docking (random seed: 405334830) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1            0          0          0
   2    0.0006341      4.515      6.924
   3    0.0006398       9.13      10.89
   4    0.0006576      7.971      10.97
   5    0.0007108      4.284      6.269
   6    0.0007238      9.897      12.58
   7    0.0007288      13.07       15.1
   8    0.0008253      4.848      7.992
   9    0.0008259      5.535      7.289
  10    0.0008259      8.876      11.82
  11    0.0008358      4.245      6.765
  12    0.0008406      5.352      8.388
  13    0.0008406      6.167      9.168
  14    0.0008957      5.821       8.33
  15    0.0009365      3.484      6.906
  16     0.001109      5.854      7.787
  17     0.001114      5.556 

8      8.181
  18       -6.287      2.591      3.139
  19       -6.254      4.315       9.27
  20        -6.11      3.155      9.794
Computing Vina grid ... done.
Performing docking (random seed: 2007989085) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1            0          0          0
   2    0.0001085      4.149      6.334
   3    0.0001503      8.095      9.222
   4    0.0001523      3.283       5.86
   5    0.0001765      5.115      5.888
   6    0.0002796      5.607      8.345
   7    0.0007493       4.27      8.253
   8    0.0009403      4.218      8.162
   9    0.0009436      5.648      9.317
  10    0.0009674       3.43      6.115
  11     0.001093      4.883      8.481
  12      0.00119      4.194      9.234
  13     0.001304      4.

Performing docking (random seed: -1374146825) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1       -7.874          0          0
   2       -7.601      1.424      1.706
   3       -7.597      1.837      2.582
   4       -7.343      2.975      3.902
   5        -7.24      2.851      4.193
   6       -7.064      3.824      6.325
   7       -7.057      2.379      3.012
   8       -7.043      1.826      2.504
   9       -6.938      3.007      3.819
  10       -6.916      2.157      7.353
  11        -6.76      2.523      7.336
  12       -6.729      3.184      4.497
  13       -6.707      2.884      3.823
  14       -6.649      2.492      6.892
  15       -6.618      3.479      7.648
  16       -6.603      3.722      7.079
  17       -6.594      4.62

 5.655      7.224
  18     0.001012      8.186      9.845
  19     0.001234      8.591      11.32
  20     0.001532      4.222      7.076
Computing Vina grid ... done.
Performing docking (random seed: -281325394) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1       -8.863          0          0
   2       -8.286       1.86      2.878
   3       -8.217      3.121       8.95
   4       -7.891      3.428      9.308
   5       -7.525      2.868      4.268
   6       -7.504      2.337      3.648
   7       -7.208      2.347      3.162
   8       -7.183      1.851      2.512
   9        -7.14      2.384      2.745
  10       -6.953      3.692      5.378
  11       -6.902      3.514      8.404
  12       -6.612        3.2       4.72
  13       -6.556   

...

Performing docking (random seed: -1503024302) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1            0          0          0
   2    6.362e-05      8.346       10.6
   3    9.648e-05        5.5      8.222
   4    0.0001072      9.494      14.13
   5    0.0001887      4.949      7.155
   6    0.0002091      3.033      8.156
   7    0.0002907      7.568      11.76
   8    0.0002922      6.393      7.984
   9    0.0002951       9.84      12.61
  10    0.0002987      6.271      9.518
  11    0.0003041      9.502      13.91
  12    0.0003107      5.016      7.031
  13    0.0003133      4.501      6.492
  14    0.0003271      6.618      9.202
  15    0.0004074      6.069      10.96
  16     0.000423      4.393      5.442
  17    0.0006033      3.59

done.
Performing docking (random seed: 676982508) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1            0          0          0
   2     0.001074      9.148      9.976
   3     0.001166      9.672      12.82
   4     0.001584      4.586      7.368
   5     0.001618       5.07       7.47
   6     0.001641      2.954      7.243
   7     0.001649      10.95      13.97
   8     0.001787      5.445      7.965
   9      0.00192      3.328      6.105
  10     0.002113      3.776      8.491
  11      0.00229      5.128      8.048
  12     0.002622      6.846      10.55
  13     0.003124      6.279      8.593
  14     0.003146      8.697      10.69
  15     0.003198      3.478      7.441
  16     0.003213      3.803      5.638
  17      0.00323      

Performing docking (random seed: 1135126107) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1            0          0          0
   2            0      5.564      8.004
   3      0.00044        8.2      12.02
   4    0.0004413      10.43       12.1
   5    0.0004421      7.796      11.69
   6    0.0004511      6.254      10.02
   7    0.0004511      4.889      8.819
   8    0.0004525      5.569      9.126
   9    0.0005835      6.677      9.572
  10     0.001245      4.011      9.195
  11     0.001246       8.11      10.93
  12      0.00152      5.792      8.436
  13     0.001522      7.057      11.16
  14     0.001522      10.03      12.15
  15     0.001527      2.862      6.847
  16     0.001529      6.314       9.54
  17     0.001532      4.954

: 

: 